# 데이터 셋업 노트북 (최초 1회 실행)

**실행 순서**: 셀을 위에서 아래로 순서대로 실행하세요.

```
Step 0: 환경 설정 (항상 먼저)
Step 1: 원본 DB 다운로드 → /content/ 로컬
Step 2: 합성 데이터셋 생성
Step 3: Drive에 tar.gz 단일 파일로 저장
```

> ⚠️ **커널 재시작 후에는 Step 0부터 다시 실행하세요.**
>
> ⚠️ 레포가 **Private**이면 GitHub Personal Access Token이 필요합니다.
>   Runtime → Manage secrets → `GITHUB_TOKEN` 추가

---
## Step 0. 환경 설정 (항상 가장 먼저 실행)

> 이 셀 하나에서 레포 클론, `sys.path` 등록, 패키지 설치를 모두 처리합니다.

In [ ]:
import os, sys, subprocess

# ── 레포 설정 ──
GITHUB_USER = 'sungmin-park-dev'
REPO_NAME   = 'tactical-speech-enhancement'
REPO_DIR    = f'/content/{REPO_NAME}'
DRIVE_ROOT  = '/content/drive/MyDrive/tactical-speech-enhancement'
COLAB_CFG   = '/content/data_config_colab.yaml'

# ── 1) 레포 클론 (Private이면 PAT 필요) ──
if not os.path.isdir(REPO_DIR):
    # 시크릿에서 토큰 읽기 (없으면 Public URL 사용)
    token = os.getenv('GITHUB_TOKEN', '')
    if token:
        clone_url = f'https://{token}@github.com/{GITHUB_USER}/{REPO_NAME}.git'
    else:
        clone_url = f'https://github.com/{GITHUB_USER}/{REPO_NAME}.git'
    print(f'클론 중: github.com/{GITHUB_USER}/{REPO_NAME}')
    result = subprocess.run(['git', 'clone', clone_url, REPO_DIR],
                            capture_output=True, text=True)
    if result.returncode != 0:
        print(f'❌ git clone 실패 (exit {result.returncode})')
        print(f'   stderr: {result.stderr.strip()}')
        if not token:
            print('\n💡 레포가 Private이면 토큰이 필요합니다:')
            print('   Runtime → Manage secrets → GITHUB_TOKEN 추가')
        raise RuntimeError('git clone 실패. 위 안내를 확인하세요.')
    print('✅ 레포 클론 완료')
else:
    subprocess.run(['git', '-C', REPO_DIR, 'pull', '--quiet'], check=True)
    print('✅ 레포 최신화 완료')

# ── 2) sys.path 등록 ──
if REPO_DIR not in sys.path:
    sys.path.insert(0, REPO_DIR)

# ── 3) 패키지 설치 ──
subprocess.run(
    [sys.executable, '-m', 'pip', 'install', '-q',
     'soundfile', 'librosa', 'pesq', 'pystoi', 'pyyaml', 'tqdm'],
    check=True
)

# ── 4) 환경 확인 ──
import torch
print(f'\nGPU: {torch.cuda.get_device_name(0) if torch.cuda.is_available() else "없음"}')
print(f'파일 목록:'); os.listdir(REPO_DIR) and None
print('\n✅ Step 0 완료. 이후 셀을 순서대로 실행하세요.')

---
## Step 1. 원본 DB 다운로드 → `/content/` 로컬

> Drive를 **거치지 않고** Colab 로컬 디스크에 직접 다운로드 (I/O 할당량 절약)

| DB | 크기 | 예상 시간 |
|---|---|---|
| LibriSpeech train-clean-100 | ~11GB 해제 | ~10분 |
| MUSAN | ~1.2GB | ~2분 |
| DEMAND (4환경) | ~400MB | ~3분 |

In [ ]:
# LibriSpeech train-clean-100
!mkdir -p /content/data/raw

if not os.path.exists('/content/data/raw/LibriSpeech/train-clean-100'):
    print('LibriSpeech 다운로드 중... (~10분)')
    !cd /content/data/raw \
        && wget -q --show-progress https://www.openslr.org/resources/12/train-clean-100.tar.gz \
        && tar -xzf train-clean-100.tar.gz \
        && rm train-clean-100.tar.gz
else:
    print('LibriSpeech: 이미 있음 — 스킵')

!find /content/data/raw/LibriSpeech -name '*.flac' | wc -l | xargs echo 'LibriSpeech flac 파일:'

In [ ]:
# MUSAN
if not os.path.exists('/content/data/raw/MUSAN/noise'):
    print('MUSAN 다운로드 중... (~2분)')
    !cd /content/data/raw \
        && wget -q --show-progress http://www.openslr.org/resources/17/musan.tar.gz \
        && tar -xzf musan.tar.gz \
        && mv musan MUSAN \
        && rm musan.tar.gz
else:
    print('MUSAN: 이미 있음 — 스킵')

!find /content/data/raw/MUSAN/noise -name '*.wav' | wc -l | xargs echo 'MUSAN noise 파일:'

In [ ]:
# DEMAND (4개 환경)
!mkdir -p /content/data/raw/DEMAND

DEMAND_ENVS = {
    'DKITCHEN': 'https://zenodo.org/record/1227121/files/DKITCHEN_16k.zip',
    'OOFFICE':  'https://zenodo.org/record/1227121/files/OOFFICE_16k.zip',
    'TBUS':     'https://zenodo.org/record/1227121/files/TBUS_16k.zip',
    'TCAR':     'https://zenodo.org/record/1227121/files/TCAR_16k.zip',
}
for name, url in DEMAND_ENVS.items():
    if os.path.exists(f'/content/data/raw/DEMAND/{name}'):
        print(f'  {name}: 이미 있음')
        continue
    print(f'  {name} 다운로드...')
    !wget -q --show-progress {url} -O /tmp/{name}.zip \
        && unzip -q /tmp/{name}.zip -d /content/data/raw/DEMAND/ \
        && rm /tmp/{name}.zip
    print(f'  {name} 완료')

!find /content/data/raw/DEMAND -name '*.wav' | wc -l | xargs echo 'DEMAND wav 파일:'
!df -h /content | tail -1

---
## Step 2. 합성 데이터셋 생성

In [ ]:
# Colab 경로로 config 생성
import yaml

with open(f'{REPO_DIR}/configs/data_config.yaml') as f:
    cfg = yaml.safe_load(f)

cfg['paths'].update({
    'librispeech_root': '/content/data/raw/LibriSpeech',
    'demand_root':      '/content/data/raw/DEMAND',
    'musan_root':       '/content/data/raw/MUSAN',
    'output_root':      '/content/data/processed',
})
with open(COLAB_CFG, 'w') as f:
    yaml.dump(cfg, f, allow_unicode=True)
print('config 저장:', COLAB_CFG)

In [ ]:
# LibriSpeech 화자 기준 분할
import numpy as np
from data.noise_mixer import scan_audio_files
from data.pipeline import split_by_speaker

all_files = scan_audio_files(
    '/content/data/raw/LibriSpeech', extensions=('.flac', '.wav')
)
print(f'총 {len(all_files)}개 파일')

split_files = split_by_speaker(all_files, train_ratio=0.70, val_ratio=0.15, seed=42)

In [ ]:
# 데이터셋 생성 실행
# 크기 조정: 빠른 검증은 N_TRAIN=500, N_VAL=100, N_TEST=100
from data.pipeline import DatasetBuilder

N_TRAIN, N_VAL, N_TEST = 5000, 1000, 1000

for env in ['military', 'general']:
    for split, n in [('train', N_TRAIN), ('val', N_VAL), ('test', N_TEST)]:
        print(f'\n=== {env}/{split} ({n}개) ===')
        builder = DatasetBuilder.from_config(
            config_path=COLAB_CFG,
            env=env,
            speech_files=split_files[split],
            rng=np.random.default_rng(42),
        )
        builder.build(n_samples=n, split=split, verbose=True)

!du -sh /content/data/processed/*
print('\n✅ 데이터셋 생성 완료')

---
## Step 3. Drive에 단일 tar.gz로 저장

> **Google 권장**: 파일을 개별 복사하지 말고 단일 압축 파일로 Drive에 전송

In [ ]:
from google.colab import drive
drive.mount('/content/drive')

DRIVE_DATA = f'{DRIVE_ROOT}/data'
!mkdir -p {DRIVE_DATA}

In [ ]:
DRIVE_ARCHIVE = f'{DRIVE_DATA}/processed_data.tar.gz'
LOCAL_ARCHIVE = '/tmp/processed_data.tar.gz'

if os.path.exists(DRIVE_ARCHIVE):
    print(f'이미 존재: {DRIVE_ARCHIVE}')
    print('강제 덮어쓰기: 다음 셀 실행')
else:
    print('① 로컬에서 압축 중...')
    !tar -czf {LOCAL_ARCHIVE} -C /content/data processed
    !du -sh {LOCAL_ARCHIVE}

    print('② Drive로 단일 파일 복사 중...')
    !cp {LOCAL_ARCHIVE} {DRIVE_ARCHIVE}
    !rm {LOCAL_ARCHIVE}

    !ls -lh {DRIVE_DATA}/
    print('\n✅ 완료! 이후 세션은 train_colab.ipynb 에서 Drive 복원 후 훈련')